In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import scanpy as sc
import matplotlib.gridspec as gridspec

 

In [2]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
#%matplotlib inline
sc.settings.figdir = "fig3"
sc.settings.set_figure_params(dpi_save=300, facecolor="white", frameon=False, figsize=(8,8))


In [3]:
PATH = '/nfs/team298/ls34/adult_skin/final_adatas/adata_combined_new.h5ad.final.filtered.addrelapse'
adata_5k=sc.read_h5ad(PATH)
NICHE_NAME ="niche19"



In [4]:
adata_5k=adata_5k[adata_5k.obs['niche19']!="Nonspecific/folded"]

In [5]:
adata_5k_all = adata_5k


In [6]:
# adata_5k = adata_5k[adata_5k.obs["Site_status"].isin(["Lesional","Non-lesional"])]


In [7]:
adata_5k_all = adata_5k_all[~adata_5k_all.obs["Site_status"].isin(["Psoriasis_replicate_PostRx", "Healthy",
                                           "Week 8 Psoriasis", "Psoriasis_replicate_non-lesional",
                                            "Non-lesional",
                                                       "Day 14_HF",
                                           ])]
adata_5k_all.obs["Site_status"].value_counts()


Site_status
Lesional                             542474
3D_Week12                            179026
3D_Lesional_baseline                 153479
Week 12                              149279
Remission_longterm (past)            140285
Remission_longterm (never)           127032
Relapse                              102198
PostRx                                96485
Remission_longterm (week 8)           71279
Psoriasis_replicate_Lesional          60640
PostRx-Lesional Dupilumab (16 wk)      6694
PostRx-Lesional Dupilumab (1 yr)       4226
Name: count, dtype: int64

NameError: name 's' is not defined

# week 12

In [ ]:
adata_5k = adata_5k_all[adata_5k_all.obs["Site_status"].isin(["Week 12", 
                                                              "3D_Week12",
                                          
                                           ])]
adata_5k_i.obs["Site_status"].value_counts()


In [ ]:
# adata_5k_obs.obs["Site_status"]

In [ ]:
#adata_5k = adata_5k_all[adata_5k_all.obs["Site_status"].isin(["Lesional","Non-lesional", "Week 12"])]
#adata_5k=adata_5k[adata_5k.obs["Site_status"]!="Lesional"]


In [ ]:
#sc.settings.figdir = "../fig2/fig2"


In [ ]:
for x in ["AD"]:
    adata_5k_i=adata_5k[adata_5k.obs["disease_overall"]==x]
    # Step 1: Contingency table of 'niche12' and 'Site_status_binary2'
    contingency = pd.crosstab(adata_5k_i.obs[NICHE_NAME], adata_5k_i.obs["Site_status"])

    # Step 2: Convert counts to proportions (row-wise)
    proportions = contingency.div(contingency.sum(axis=1), axis=0)

    # Step 3: Compute baseline proportions across all niche12 (i.e., global average)
    total_counts = adata_5k_i.obs["Site_status"].value_counts(normalize=True)
    total_counts
    
    
    
    baseline_df = pd.DataFrame([total_counts], index=["Overall"])

# # Step 4: Sort niches by proportion of "Lesional"
# if "Lesional" not in proportions.columns:
#     raise ValueError("Column 'Lesional' not found in Site_status_binary2")

    sorted_idx = proportions.sort_values("Week 12", ascending=True).index

    # Step 5: Concatenate baseline + sorted niches
    final_df = pd.concat([baseline_df, proportions.loc[sorted_idx]], axis=0)

    # Step 6: Ensure consistent column order
    desired_col_order = ["Week 12", "Non-lesional"]
    final_df = final_df[desired_col_order]

    # Step 7: Plotting
    fig, ax = plt.subplots(figsize=(10, 2), dpi=300)

    final_df.plot(
        kind='bar',
        stacked=True,
        color=["#FFF59D", "#b0e0f0"],
        edgecolor='black',
        linewidth=0.2,
        ax=ax
    )

    # Fix xtick alignment manually
    ax.set_xticks(range(len(final_df)))
    ax.set_xticklabels(final_df.index, rotation=90, ha='center')

    # Add dashed horizontal line at the baseline proportion of "Lesional"
    lesional_label = "Lesional"
    if lesional_label in total_counts:
        baseline_value = total_counts[lesional_label]
        ax.axhline(y=baseline_value, color='black', linestyle='--', linewidth=2)

    # Aesthetic clean-up
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlabel('')
    ax.set_ylabel('Proportion')
    
    # Add dashed horizontal line at the baseline proportion of "Lesional"
    lesional_label = "Week 12"
    if lesional_label in total_counts:
        baseline_value = total_counts[lesional_label]
        ax.axhline(y=baseline_value, color='black', linestyle='--', linewidth=2)

    # Clean legend
    legend = ax.legend(title='Site_status_binary2', bbox_to_anchor=(1.05, 1), loc='upper left')
    legend.get_frame().set_linewidth(0.0)  # Remove legend box

    ax.grid(False)
    ax.set_title('')

    plt.tight_layout()
    plt.savefig("9a.pdf", dpi=300, bbox_inches="tight")

    plt.show()